# **PHASE 02 — DATA PREPROCESSING**
- normalize column names for better understanding and model standards

Company Name| Symbol| Share Volume| Trade Volume| Previous Close (Rs.)| Open (Rs.)| High (Rs.)|Low (Rs.)| **Last Trade (Rs.)| Change (Rs.)| Change (%)

##### Before modeling, you should now write a schema mapper that converts this raw exchange format → standardized ML format
Last Trade	    >  close<br>
Previous Close	>  prev_close<br>
Share Volume	>  volume<br>
Trade Volume	>  trades<br>

### **Table flow**<br> 
#### stocks_table -> stocks_clean_table -> stocks_sorted_table -> stocks_features_clean_table -> stocks_features_labeled_table 


## **Data Hndling with DuckDB**

### Why Data Engineers Prefer DuckDB
- Real data pipelines often use it for:
- stock market analysis
- log processing
- machine learning datasets
- financial backtesting
- Because it handles millions to billions of rows locally
#### What we do here
- Creating the connection to DuckDB
- Creating a database named "cse_data.db"
- Craeting relvent tablesd to store data
- Read csv cin folder & add them into table
- Ensure data quality & Data cleaning
- Standardization the columns names
- Sorting
- Indexing(symbol &  date) for speed data accessing
- exporting final table as **MASTER_CSE_DATA.csv**

### Creating log file

In [32]:
# ================ CONFIG ===================

os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit

# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

### Create DuckDB Connection

In [33]:
import os
import duckdb

# Database folder
DB_FOLDER = "database"
os.makedirs(DB_FOLDER, exist_ok=True)
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")
con = duckdb.connect(database=DB_PATH)  # Create/connect DB

### HELPERS

In [34]:
# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

In [35]:
import duckdb

con = duckdb.connect(database=DB_PATH)  # Create/connect DB

# ================= CREATE STOCKS TABLE =================

log("Creating DuckDB table 'stocks_table' ...")

con.execute("""
    CREATE TABLE IF NOT EXISTS stocks_table AS SELECT
        TRIM(company) AS company,
        TRIM(symbol) AS symbol,
    
        CAST(volume AS INTEGER) AS volume,
        CAST(trades AS INTEGER) AS trades,
        
        CAST(prev_close AS DOUBLE) AS prev_close,
        CAST(open AS DOUBLE) AS open,
        CAST(high AS DOUBLE) AS high,
        CAST(low AS DOUBLE) AS low,
        CAST(close AS DOUBLE) AS close,
    
        CAST(change AS DOUBLE) AS change,
        CAST(change_percentage AS DOUBLE) AS change_percentage,
        
        CAST(date AS DATE) AS date
        
    FROM raw_stocks_table

    WHERE
    symbol IS NOT NULL
    AND TRIM(symbol) != ''
    AND close IS NOT NULL
    AND CAST(close AS DOUBLE) > 0
    AND (volume IS NULL OR CAST(volume AS INTEGER) >= 0)
""")

log("stocks_table created")



# ================  REMOVE DUPLICATES =====================

# Removes duplicate rows for the same symbol and date
# numbers duplicate rows starting from 1
# WHERE rn = 1 → keeps only the first row of each symbol/date combination

log("Removing duplicates...")

# Creates a table named stocks_clean_table
# If it already exists → deletes old one and recreates it
# add ne column dup_index, assign occurnce counnt for each record only keeep occurence number 1 others will be delted
# 
con.execute("""
    CREATE OR REPLACE TABLE stocks_clean_table AS
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY symbol, date
                   ORDER BY date
               ) AS dup_index
        FROM stocks_table
    )
    WHERE dup_index = 1
""")



# ================ SORT ========================

log("Sorting dataset...")

# Create new table that cotain sorted data by symbol then date
con.execute("""
    CREATE OR REPLACE TABLE stocks_sorted_table AS
    SELECT *
    FROM stocks_clean_table
    ORDER BY symbol, date
""")



# ================ CREATE INDEX ========================

log("Creating composite index on symbol+date for speed...")

# Adds a composite index on (symbol, date)
# Speeds up all queries that filter by symbol, date, or both
con.execute("""
    CREATE INDEX IF NOT EXISTS idx_symbol_date ON stocks_sorted_table(symbol, date)
""")



# ================ MISSING DATE REPORT ==================
# TODO : if needed



# ================ EXPORT =======================

con.execute(f"""
    COPY stocks_sorted_table TO '{MASTER_CSV_OUTPUT}' (HEADER, DELIMITER ',')
""")

log(f"MASTER DATASET CREATED → {MASTER_CSV_OUTPUT}")


# ================= PREVIEW =================

stocks_table_df = con.execute("""
    SELECT *
    FROM stocks_table
    LIMIT 5
""").fetchdf()

print(stocks_table_df)

log("=== PROCESSING FINISHED ===")

# stocks_table
# stocks_clean_table
# stocks_sorted_table

Creating DuckDB table 'stocks_table' ...
stocks_table created
Removing duplicates...
Sorting dataset...
Creating composite index on symbol+date for speed...
MASTER DATASET CREATED → output_csv\MASTER_CSE_DATA.csv
                                company      symbol    volume  trades  \
0      INDUSTRIAL ASPHALTS (CEYLON) PLC  ASPH.N0000  14032048     177   
1        RADIANT GEMS INTERNATIONAL PLC  RGEM.N0000     76307     280   
2  LAKE HOUSE PRINTERS & PUBLISHERS PLC  LPRT.N0000      2384     152   
3                     BAIRAHA FARMS PLC   BFL.N0000    249279    1254   
4      KERNER HAUS GLOBAL SOLUTIONS PLC  CPRT.N0000      1322     112   

   prev_close    open   high     low   close  change  change_percentage  \
0        0.40    0.50    0.5    0.40    0.50    0.10              25.00   
1       83.60   88.00  104.5   86.00  104.00   20.40              24.40   
2      475.75  480.00  594.5  480.00  591.00  115.25              24.22   
3      384.75  409.75  470.0  409.75  458.25   7

### Test speed of query execution with indexing

In [36]:
# commnet above code's indexing part and give a dofferent name for db cvvration try it with below code
# try agian with uncommenting

import time
import duckdb

# Change this to the DB you want to test
con = duckdb.connect(DB_PATH)  # or 'cse_data_index.db'

start = time.time()
results = con.execute("""
    SELECT * FROM stocks_sorted_table WHERE symbol='TAJ.N0000'
""").fetchall()
end = time.time()

print(f"Rows returned: {len(results)}")
print(f"Query time: {end - start:.4f} seconds")

Rows returned: 60
Query time: 0.0034 seconds


In [37]:
columns_df = con.execute("""
    DESCRIBE stocks_table
""").fetch_df()
print(columns_df)

#Index(['company', 'symbol', 'volume', 'trades', 'prev_close', 'open', 'high',
#       'low', 'close', 'change', 'change_percentage', 'date'],
 #     dtype='object')

          column_name column_type null   key default extra
0             company     VARCHAR  YES  None    None  None
1              symbol     VARCHAR  YES  None    None  None
2              volume     INTEGER  YES  None    None  None
3              trades     INTEGER  YES  None    None  None
4          prev_close      DOUBLE  YES  None    None  None
5                open      DOUBLE  YES  None    None  None
6                high      DOUBLE  YES  None    None  None
7                 low      DOUBLE  YES  None    None  None
8               close      DOUBLE  YES  None    None  None
9              change      DOUBLE  YES  None    None  None
10  change_percentage      DOUBLE  YES  None    None  None
11               date        DATE  YES  None    None  None


### Close DuckDB Connection

In [38]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory
        
# stocks_sorted_table

DuckDB connection closed.


# Note for columns

- **future_close_10d** : actual closing price 10 days after the current date

# 1️⃣ Core Algorithm: Gradient Boosted Trees (XGBoost / LightGBM) DONE

Perfect — now we’re talking **“money-first, best-possible model”** mode. Forget simplicity; we want **maximum predictive edge for CSE stocks**. Here’s how to do it properly:

---

# 🏆 Best-Performing Model Setup for Your Table

### **1️⃣ Core Algorithm: Gradient Boosted Trees (XGBoost / LightGBM)**

* Tree-based, handles tabular features perfectly
* Captures **non-linear relationships** and interactions in momentum, liquidity, volatility
* Produces **probabilities** → essential for Top-K trade picks

**Why not Random Forest or Neural Nets?**

* RF → good, but ceiling lower
* Neural Nets → overkill, unstable on small tabular data, need tons of training → overfit easily
* SVMs → slow, scale poorly for millions of rows

---

### **2️⃣ Ensemble for Extra Edge**

* XGBoost (main)
* Random Forest (sanity check / reduce false positives)
* Logistic Regression (baseline filter for extreme outliers)

> Voting or weighted average gives **robust predictions**, reduces risk of bad trades.

---

### **3️⃣ Feature Set (Max Signal)**

* Momentum: ret_5, ret_10, trend slope, range_position
* Volatility: std_close_10, ATR, drawdown_20
* Liquidity: volume_ratio, volume_z, liquidity_rank
* Relative Strength: rel_ret_5d_vs_ASPI, sector_rel_ret_5d
* Event Signals: pre_dividend_flag, days_to_dividend, rights_issue_flag

**Tip:** The more **high-signal, engineered features**, the better the trees perform.

---

### **4️⃣ Model Tuning**

* Use **walk-forward validation** → avoids overfit to historical data
* Optimize hyperparameters:

  ```
  n_estimators: 300–500
  learning_rate: 0.05–0.1
  max_depth: 3–7
  subsample: 0.7–0.9
  colsample_bytree: 0.7–0.9
  min_child_weight: 1–5
  ```
* Adjust probability threshold for **Top-K picks** → don’t just take 0.5 cutoff

---

### **5️⃣ Execution Workflow**

1. Run model weekly → predict probabilities for each stock
2. Filter by liquidity / volume / price thresholds
3. Select **Top 5–10 stocks** for actual trades
4. Apply stop-loss / take-profit rules → protect capital
5. Track performance → retrain model monthly

---

### **6️⃣ Optional Advanced Tricks**

* SHAP values → understand feature contribution → remove weak/noisy features
* Ensemble voting → reduce false positives
* Include **macro & market features**: ASPI returns, turnover, volatility index

---

💡 **Bottom line:**

* The **best possible setup** for your money-focused CSE trading is:
  **XGBoost (main) + optional Random Forest sanity check + engineered features + Top-K weekly picks + walk-forward validation + probability threshold tuning**

This is **state-of-the-art for tabular, structured stock data**, and gives the **highest chance of profitable trades**.

---

If you want, I can now make a **full blueprint for your dataset**, showing:

* Exact **features + transformations**
* Model setup & hyperparameters
* Thresholds & Top-K weekly trading logic

…ready to implement step by step for **maximum returns**.

Do you want me to do that?
